# 02 — What makes *fusion* different from *human*?

A fusion text is a human text partly rewritten by an LLM. Five complementary views:

| § | Question | Data |
|---|---|---|
| A | What exactly does the LLM change? (amount, position, inserted vs deleted words) | human ↔ fusion **pairs** of the same source found by `00_make_splits` |
| B | Which style statistics separate human from fusion, per domain? | training split |
| C | Which words / punctuation are typical of each class, consistently across domains? | training split |
| D | *Where* in a text is the fusion signal: start, middle or end? | model on training split, scored on validation thirds |
| E | Where does the baseline confuse human and fusion? | baseline validation scores |

Only the training split is used to look for patterns (A uses pairs that cross train/val by construction; D and E
only *score* validation, nothing is tuned on it). Outputs are kept short on purpose.

In [ ]:
import sys
from pathlib import Path

SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [ ]:
import difflib
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from config import LABELS, OUTPUT_DIR, SEED, SPLITS_DIR
from utils import load_rows, load_splits

pd.set_option("display.width", 180)
TOKEN_PATTERN = r"(?u)\b\w+\b|[^\w\s]"      # same tokens as the baseline's word_cased_punct
TOK = re.compile(TOKEN_PATTERN)

train_df, val_df = load_splits()
hf_train = train_df[train_df.label.isin(["human", "fusion"])].reset_index(drop=True)
print(f"train={len(train_df):,} (human+fusion={len(hf_train):,})  val={len(val_df):,}")

## A. Paired view: a human source next to its fusion rewrite

`near_dup_pairs.csv` holds train/val text pairs sharing ≥20% of word 5-grams. Pairs labelled human ↔ fusion are
(very likely) the original and its partial rewrite, so a word-level diff shows *exactly* what the LLM changed.

Caveat for the report: heavily rewritten texts share fewer 5-grams and are missed, so these pairs are biased
towards *light* rewrites.

In [ ]:
pairs = pd.read_csv(SPLITS_DIR / "near_dup_pairs.csv")
pairs["kind"] = ["↔".join(sorted(t)) for t in zip(pairs.train_label, pairs.val_label)]
print("All near-duplicate pairs by label combination:")
print(pd.crosstab(pairs.domain, pairs.kind, margins=True).to_string())

hf = pairs[pairs.kind == "fusion↔human"].copy()
hf["human_id"] = np.where(hf.train_label == "human", hf.train_id, hf.val_id)
hf["fusion_id"] = np.where(hf.train_label == "fusion", hf.train_id, hf.val_id)
hf = hf.drop_duplicates(["human_id", "fusion_id"]).reset_index(drop=True)
texts = load_rows(pd.unique(pd.concat([hf.human_id, hf.fusion_id]))).set_index("id")["text"]
print(f"\nhuman↔fusion pairs used for the diff: {len(hf):,}")

In [ ]:
MAX_TOKENS = 1_500   # difflib is quadratic in the worst case; long texts are cut for the diff only


def diff(h: str, f: str):
    a, b = TOK.findall(h)[:MAX_TOKENS], TOK.findall(f)[:MAX_TOKENS]
    ops = difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes()
    return a, b, ops


rows, inserted, deleted, positions = [], Counter(), Counter(), []
for r in hf.itertuples():
    a, b, ops = diff(texts[r.human_id], texts[r.fusion_id])
    kept = sum(i2 - i1 for t, i1, i2, _, _ in ops if t == "equal")
    edits = [o for o in ops if o[0] != "equal"]
    for t, i1, i2, j1, j2 in edits:
        inserted.update(w.lower() for w in b[j1:j2])
        deleted.update(w.lower() for w in a[i1:i2])
        # position (0 = start, 1 = end of the human text) of every changed token, insertions at their anchor
        positions += [i / max(1, len(a)) for i in (range(i1, i2) if i2 > i1 else [i1])]
    rows.append({"domain": r.domain, "jaccard": r.jaccard, "human_tokens": len(a),
                 "kept_share": kept / max(1, len(a)), "n_edit_spans": len(edits),
                 "len_ratio": len(b) / max(1, len(a))})
pair_stats = pd.DataFrame(rows)

print("Per pair (median [IQR]):")
q = pair_stats.drop(columns="domain").quantile([.25, .5, .75]).T.round(3)
print(q.to_string())
print("\nMedian per domain:")
print(pair_stats.groupby("domain")[["kept_share", "n_edit_spans", "len_ratio"]].median().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(pair_stats.kept_share, bins=30)
axes[0].set(title="Share of human tokens kept unchanged", xlabel="kept share", ylabel="pairs")
axes[1].hist(pair_stats.len_ratio.clip(0, 2), bins=30)
axes[1].set(title="Length ratio fusion / human", xlabel="ratio (clipped at 2)")
axes[2].hist(positions, bins=20, density=True)
axes[2].set(title="Where edits happen in the human text", xlabel="relative position (0 = start, 1 = end)")
plt.tight_layout(); plt.show()

**What the LLM adds and removes.** Log-ratio of each token's frequency among *inserted* vs *deleted* tokens
(add-one smoothing, tokens seen ≥ 5 times).

In [ ]:
V = len(set(inserted) | set(deleted))
n_ins, n_del = sum(inserted.values()), sum(deleted.values())
lr = pd.Series({w: np.log((inserted[w] + 1) / (n_ins + V)) - np.log((deleted[w] + 1) / (n_del + V))
                for w in set(inserted) | set(deleted) if inserted[w] + deleted[w] >= 5}).sort_values()
show = lambda s: ", ".join(f"{w!r}({inserted[w]}/{deleted[w]})" for w in s.index)
print("Mostly ADDED by the LLM   (inserted/deleted counts):", show(lr.tail(25)[::-1]))
print("\nMostly REMOVED by the LLM (inserted/deleted counts):", show(lr.head(25)))

In [ ]:
def render(h: str, f: str, width: int = 900) -> str:
    """Human text with [-deleted-] and {+inserted+} spans, cropped around the first edit."""
    a, b, ops = diff(h, f)
    out = []
    for t, i1, i2, j1, j2 in ops:
        if t == "equal":
            out.append(" ".join(a[i1:i2]))
        else:
            if i2 > i1: out.append("[-" + " ".join(a[i1:i2]) + "-]")
            if j2 > j1: out.append("{+" + " ".join(b[j1:j2]) + "+}")
    s = " ".join(out)
    start = max(0, s.find("[-") if "[-" in s else s.find("{+")) - 150
    return ("…" if start > 0 else "") + s[start:start + width] + "…"


ex = pair_stats.assign(i=range(len(pair_stats)))
ex = ex[ex.n_edit_spans > 0].sort_values("kept_share")   # show only pairs that were actually edited
for i in ex.i.iloc[[len(ex) // 4, len(ex) // 2, 3 * len(ex) // 4]] if len(ex) >= 4 else ex.i:
    r = hf.iloc[i]
    print(f"--- {r.domain}  kept_share={pair_stats.kept_share[i]:.2f}\n{render(texts[r.human_id], texts[r.fusion_id])}\n")

## B. Style statistics: which ones separate human from fusion?

Each statistic is computed per text. Separability is the **ROC-AUC of the raw statistic** for human (0) vs fusion (1):
0.5 = no signal, > 0.5 = higher in fusion, < 0.5 = higher in human. Computed per domain, because domains differ
in style far more than classes do (e.g. umlauts in German, newlines in blogs).

In [ ]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def style_features(s: pd.Series) -> pd.DataFrame:
    s = s.astype(str)
    words = s.str.count(r"\S+").clip(lower=1)
    sents = s.map(lambda t: [x for x in SENT_SPLIT.split(t) if x.strip()])
    sent_len = sents.map(lambda xs: [len(x.split()) for x in xs] or [0])
    per100 = lambda pat: s.str.count(pat) / words * 100
    n_curly, n_straight = s.str.count("’"), s.str.count("'")

    def half_gap(t):                       # style drift inside one text: mean word length, 1st vs 2nd half
        w = t.split(); h = len(w) // 2
        return np.nan if h < 20 else abs(np.mean([len(x) for x in w[:h]]) - np.mean([len(x) for x in w[h:]]))

    f = pd.DataFrame(index=s.index)
    f["n_words"] = words
    f["sent_len_mean"] = sent_len.map(np.mean)
    f["sent_len_cv"] = sent_len.map(lambda x: np.std(x) / max(1, np.mean(x)))   # "burstiness"
    f["ttr_first300"] = s.map(lambda t: len({w.lower() for w in t.split()[:300]}) / max(1, min(300, len(t.split()))))
    f["chars_per_word"] = s.str.replace(r"\s+", "", regex=True).str.len() / words
    f["commas_per100"] = per100(",")
    f["excl_q_per100"] = per100(r"[!?]")
    f["lower_i_per100"] = per100(r"(?<![\w'’])i(?![\w'’])")
    f["lower_sent_start"] = sents.map(lambda xs: np.mean([x.lstrip()[:1].islower() for x in xs]) if xs else 0)
    f["curly_apos_share"] = n_curly / (n_curly + n_straight).replace(0, np.nan)
    f["mixed_apostrophes"] = ((n_curly > 0) & (n_straight > 0)).astype(int)
    f["mixed_dquotes"] = (s.str.contains("[“”]") & s.str.contains('"')).astype(int)
    f["dashes_per100"] = per100("[—–]")
    f["non_ascii_per1k"] = s.str.count(r"[^\x00-\x7F]") / s.str.len().clip(lower=1) * 1000
    f["double_space"] = s.str.contains("  ", regex=False).astype(int)
    f["newlines_per100"] = per100("\n")
    f["digits_per100"] = per100(r"\d")
    f["half_wordlen_gap"] = s.map(half_gap)
    return f


feats = style_features(hf_train.text)
is_fusion = (hf_train.label == "fusion").astype(int)


def auc(col, mask):
    x, y = feats.loc[mask, col], is_fusion[mask]
    ok = x.notna()
    return np.nan if y[ok].nunique() < 2 or x[ok].nunique() < 2 else roc_auc_score(y[ok], x[ok])


domains = sorted(hf_train.domain.unique())
auc_tab = pd.DataFrame({d: {c: auc(c, hf_train.domain == d) for c in feats} for d in domains})
auc_tab.insert(0, "ALL", [auc(c, slice(None)) for c in feats])
auc_tab = auc_tab.loc[(auc_tab.drop(columns="ALL") - 0.5).abs().mean(axis=1).sort_values(ascending=False).index]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.38 * len(auc_tab) + 1.5))
im = ax.imshow(auc_tab.values, cmap="RdBu_r", vmin=0.25, vmax=0.75, aspect="auto")
ax.set_xticks(range(auc_tab.shape[1]), auc_tab.columns, rotation=35, ha="right")
ax.set_yticks(range(auc_tab.shape[0]), auc_tab.index)
for i in range(auc_tab.shape[0]):
    for j in range(auc_tab.shape[1]):
        v = auc_tab.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="AUC human(0) vs fusion(1)   red = higher in fusion")
ax.set_title("Single-feature separability of human vs fusion, per domain (training split)")
plt.tight_layout(); plt.show()

Does fusion sit *between* human and ai? For the 8 most separating statistics: overall medians per class, then
per domain the **position of the fusion median** on the human→ai scale, `(fusion − human) / (ai − human)`:
0 = like human, 1 = like ai, outside [0, 1] = fusion is more extreme than both (NaN when human ≈ ai).

In [ ]:
top8 = auc_tab.index[:8].tolist()
all_feats = style_features(train_df.text)[top8].assign(label=train_df.label.values, domain=train_df.domain.values)
print("Overall medians:")
print(all_feats.groupby("label")[top8].median().loc[LABELS].T.round(3).to_string())

med = all_feats.groupby(["domain", "label"])[top8].median()
pos = {}
for d in domains:
    h, a_, f = med.loc[(d, "human")], med.loc[(d, "ai")], med.loc[(d, "fusion")]
    gap = (a_ - h).where((a_ - h).abs() > 1e-9)
    pos[d] = (f - h) / gap
print("\nPosition of fusion between human (0) and ai (1):")
print(pd.DataFrame(pos).round(2).to_string())

## C. Words and punctuation typical of human vs fusion, consistent across domains

Log-odds ratio with an informative Dirichlet prior (Monroe et al., 2008, *"Fightin' Words"*), computed **per
domain**, then averaged. A token ranks high only if it leans the same way in many domains, so domain-specific
vocabulary (which only reflects class balance per domain) does not dominate.

In [ ]:
cv = CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, ngram_range=(1, 2), min_df=10, max_features=100_000)
X = cv.fit_transform(hf_train.text.str.slice(0, 4000))     # training rows only
vocab = cv.get_feature_names_out()


def log_odds_z(Xa, Xb):
    ya, yb = np.asarray(Xa.sum(0)).ravel(), np.asarray(Xb.sum(0)).ravel()
    alpha = ya + yb + 0.01                                   # prior: pooled counts of both classes
    na, nb, a0 = ya.sum(), yb.sum(), alpha.sum()
    delta = np.log((ya + alpha) / (na + a0 - ya - alpha)) - np.log((yb + alpha) / (nb + a0 - yb - alpha))
    return delta / np.sqrt(1 / (ya + alpha) + 1 / (yb + alpha))


Z = {}
for d in domains:
    m = (hf_train.domain == d).to_numpy()
    fu, hu = m & (is_fusion.to_numpy() == 1), m & (is_fusion.to_numpy() == 0)
    if fu.sum() >= 50 and hu.sum() >= 50:
        Z[d] = log_odds_z(X[fu], X[hu])                     # > 0: more typical of fusion
Z = pd.DataFrame(Z, index=vocab)
summary = pd.DataFrame({"mean_z": Z.mean(axis=1), "domains_same_sign": (np.sign(Z).eq(np.sign(Z.mean(axis=1)), axis=0)).sum(axis=1)})
print(f"domains used: {list(Z.columns)}")
for name, part in [("FUSION", summary.sort_values("mean_z", ascending=False)),
                   ("HUMAN", summary.sort_values("mean_z"))]:
    top = part.head(25)
    print(f"\nMost {name}-typical: " + ", ".join(f"{t!r}({s}/{Z.shape[1]})" for t, s in zip(top.index, top.domains_same_sign)))

## D. Where in the text is the fusion signal?

A human-vs-fusion logistic regression (word + punctuation TF-IDF) is trained on the **training split** and then
scores **validation** texts cut into thirds. If the LLM rewrites mostly one part, that third separates best. A
text-level detector that sees only the first 512 tokens (the encoder's limit) would miss signal that sits late.

In [ ]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, ngram_range=(1, 2), min_df=3,
                      max_features=200_000, sublinear_tf=True, dtype=np.float32)
clf = LogisticRegression(C=4, max_iter=2000, class_weight="balanced")
clf.fit(vec.fit_transform(hf_train.text.str.slice(0, 4000)), is_fusion)

hf_val = val_df[val_df.label.isin(["human", "fusion"])].reset_index(drop=True)
y_val = (hf_val.label == "fusion").astype(int)


def part(t: str, k: int, n: int = 3) -> str:
    w = t.split(); s = len(w) // n
    return " ".join(w[k * s:(k + 1) * s] if k < n - 1 else w[k * s:])


scores = {"whole text (first 4k chars)": hf_val.text.str.slice(0, 4000)}
scores.update({name: hf_val.text.map(lambda t, k=k: part(t, k)) for k, name in enumerate(["first third", "middle third", "last third"])})
auc_parts = {}
for name, txt in scores.items():
    p = clf.predict_proba(vec.transform(txt))[:, 1]
    auc_parts[name] = {"ALL": roc_auc_score(y_val, p),
                       **{d: roc_auc_score(y_val[m], p[m]) for d in domains
                          if (m := (hf_val.domain == d).to_numpy()).sum() > 50 and y_val[m].nunique() == 2}}
print("Validation AUC, human vs fusion, by text part:")
print(pd.DataFrame(auc_parts).T.round(3).to_string())

## E. Baseline confusions between human and fusion

Uses the 3-class baseline's saved validation scores (`outputs/val_scores/01_tfidf_best.csv`).

In [ ]:
vs = pd.read_csv(OUTPUT_DIR / "val_scores" / "01_tfidf_best.csv").set_index("id").loc[val_df.id]
pred = vs.idxmax(axis=1).to_numpy()
ev = val_df[["domain", "label"]].assign(pred=pred)
rate = lambda g, a, b: ((g.label == a) & (g.pred == b)).sum() / max(1, (g.label == a).sum())
conf = pd.DataFrame({d: {"n_human": int((g.label == "human").sum()), "n_fusion": int((g.label == "fusion").sum()),
                         "human→fusion": rate(g, "human", "fusion"), "fusion→human": rate(g, "fusion", "human"),
                         "fusion→ai": rate(g, "fusion", "ai"), "ai→fusion": rate(g, "ai", "fusion")}
                     for d, g in ev.groupby("domain")}).T
print(conf.round(3).sort_values("fusion→human", ascending=False).to_string())

## Findings (write in your own words after running)

1. **What the LLM changes** (A): share kept, number and position of edits, what it adds/removes.
2. **Style signals** (B): which statistics separate human from fusion, and in which domains; is fusion between
   human and ai?
3. **Vocabulary** (C): tokens consistently typical of each class.
4. **Position** (D): is the signal spread out or concentrated? What does that imply for 512-token truncation?
5. **Where the baseline fails** (E): which domains, and does that match A–D?
6. **What this changes in the modelling plan:** features to add, long-text handling, per-domain thresholds.